# **Recurrent Neural Network**

### Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

### Load Pre-Processed Features

In [ ]:
PROCESSED_DIR = "processed_data"

# Load features, labels and folds
X_rnn = np.load(os.path.join(PROCESSED_DIR, "X_rnn_mfcc.npy"), allow_pickle=True)
y = np.load(os.path.join(PROCESSED_DIR, "y_labels.npy"))
folds = np.load(os.path.join(PROCESSED_DIR, "folds.npy"))

## **Model Architecture**

In [ ]:
n_mfcc = X_rnn[0].shape[1]
n_classes = len(np.unique(y))

*Unidirectional LSTM*

In [ ]:
def rnn_model(n_mfcc, n_classes):
    model = models.Sequential()

    model.add(
        layers.LSTM(128, return_sequences=True, input_shape=(None, n_mfcc))
    )

    model.add(layers.LSTM(64))

    model.add(layers.Dense(64, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(n_classes, activation="softmax"))

    return model

model = rnn_model(n_mfcc=25, n_classes=n_classes)
model.summary()

*Bidirectional LSTM*

In [ ]:
def rnn_model(n_mfcc, n_classes):
    model = keras.Sequential()

    model.add(
        layers.Bidirectional(layers.LSTM(128, return_sequences=True), input_shape=(None, n_mfcc))
    )

    model.add(
        layers.Bidirectional(layers.LSTM(64))
    )

    model.add(layers.Dense(64, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(n_classes, activation="softmax"))

    return model

model = rnn_model(n_mfcc=25, n_classes=n_classes)
model.summary()

## **Training Strategy**

In [ ]:
learning_rate = 1e-3
batch_size = 32

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5
)

callback_list = [early_stop, reduce_lr]

## **Performance Evaluation**

In [ ]:
MAX_LEN = max(seq.shape[0] for seq in X_rnn)

# Padding the sequences so that all have the same length
def pad_sequences_list(X_list, max_len=MAX_LEN):
    n_samples = len(X_list)
    X_padded = np.zeros((n_samples, max_len, n_mfcc), dtype=np.float32)
    for i, seq in enumerate(X_list):
        L = min(seq.shape[0], max_len)
        X_padded[i, :L, :] = seq[:L, :]
    return X_padded

# Confusion Matrix
conf_mat = np.zeros((n_classes, n_classes), dtype=np.int64)

# Accuracies per fold
fold_accuracies = []

# Accuracies and Loss metrics per fold
fold_histories = []

# 10 fold cross-validation
for test_fold in range(1, 11):
    
    val_fold = (test_fold % 10) + 1

    # 8 training folds
    train_idx = np.where((folds != test_fold) & (folds != val_fold))[0]
    # 1 validation fold
    val_idx = np.where(folds == val_fold)[0]
    # 1 test fold
    test_idx = np.where(folds == test_fold)[0]

    # Split features
    X_train_list = X_rnn[train_idx]
    X_val_list = X_rnn[val_idx]
    X_test_list = X_rnn[test_idx]

    # Padding the sequences
    X_train = pad_sequences_list(X_train_list, MAX_LEN)
    X_val = pad_sequences_list(X_val_list, MAX_LEN)
    X_test = pad_sequences_list(X_test_list, MAX_LEN)

    # Split labels
    y_train = y[train_idx]
    y_val = y[val_idx]
    y_test = y[test_idx]

    model = rnn_model(n_mfcc=n_mfcc, n_classes=n_classes)

    optimizer = optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    # Train
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=batch_size,
        callbacks=callback_list,
        verbose=1,
    )

    fold_histories.append(history.history)

    # Test predictions
    y_pred_probs = model.predict(X_test)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Accuracy
    fold_acc = np.mean(y_pred == y_test)
    fold_accuracies.append(fold_acc)

    # Confusion matrix for each fold
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(n_classes))
    conf_mat += cm

*Confusion Matrix*

In [ ]:
mean_acc = np.mean(fold_accuracies)
std_acc = np.std(fold_accuracies)

print("Fold accuracies:", fold_accuracies)
print(f"Mean accuracy: {mean_acc:.4f}")
print(f"Std of accuracy: {std_acc:.4f}")

class_names = ["air-conditioner","car-horn","children-playing","dog-bark","drilling","engine-idling","gun-shot","jackhammer","siren","street-music"]
plt.figure(figsize=(8, 7))

sns.heatmap(
    conf_mat,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    linecolor="black"
)

plt.title("Confusion Matrix", fontsize=16)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)

plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()


*Per-class and averaged Precision, Recall and F1 score*

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(y_true_all, y_pred_all, labels=np.arange(n_classes), average=None)

macro_precision = np.mean(precision)
macro_recall = np.mean(recall)
macro_f1 = np.mean(f1)

df = pd.DataFrame({
    "Class": class_names,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

df.loc["Macro-average"] = [
    "Macro-average",
    macro_precision,
    macro_recall,
    macro_f1
]

df

*Mean Train/Validation Accuracy and Loss*

In [ ]:
max_epochs = max(len(h["accuracy"]) for h in fold_histories)

def pad(seq, max_len):
    return seq + [seq[-1]] * (max_len - len(seq))

acc_matrix = np.array([pad(h["accuracy"], max_epochs) for h in fold_histories])
val_acc_matrix = np.array([pad(h["val_accuracy"], max_epochs) for h in fold_histories])
loss_matrix = np.array([pad(h["loss"], max_epochs) for h in fold_histories])
val_loss_matrix = np.array([pad(h["val_loss"], max_epochs) for h in fold_histories])

mean_acc = np.mean(acc_matrix, axis=0)
mean_val_acc = np.mean(val_acc_matrix, axis=0)
mean_loss = np.mean(loss_matrix, axis=0)
mean_val_loss = np.mean(val_loss_matrix, axis=0)

epochs = np.arange(1, max_epochs + 1)

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(epochs, mean_acc, label="Train accuracy")
ax1.plot(epochs, mean_val_acc, label="Validation accuracy", linestyle="--")
ax1.set_xlabel("Epochs")
ax1.set_ylabel("Accuracy")
ax1.set_ylim(0.0, 1.0)
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(epochs, mean_loss, label="Train loss", marker="x", linewidth=1)
ax2.plot(epochs, mean_val_loss, label="Validation loss", marker="x", linewidth=1, linestyle="--")
ax2.set_ylabel("Loss")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="lower right")

plt.tight_layout()
plt.show()